# 🟢 Day 3 — Branches & Loops

**LangGraph from Zero to Advanced · Day 3 of 16**

**Today's goal:** let the graph **decide** where to go next, and **repeat** steps until a job is done. This is the heart of every AI agent.

| Lesson | Topic |
|---|---|
| 1 | Conditional edge = an `if` between nodes |
| 2 | Clean routers with `Literal` |
| 3 | Routers that return labels (path map) |
| 4 | Loops: go back to an earlier node |
| 5 | Safety net: recursion limit |
| 6 | `Command`: update state AND choose the next node |
| 7 | Mini project: support-ticket triage |
| 8 | Practice |

No API key needed today.

In [ ]:
%pip install -q -U langgraph

In [ ]:
import operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## Lesson 1 — Conditional edge = an `if` between nodes

A **router** is a function that reads the state and returns **the name of the next node**.

```
                ┌──► even ──┐
START ──► check ┤           ├──► END
                └──► odd  ──┘
```

`add_conditional_edges("check", router)` means: *after `check`, call `router` to decide.*

In [ ]:
class NumberState(TypedDict):
    number: int
    result: str

def check(state):
    return {}                      # does nothing — it's just a decision point

def even(state):
    return {"result": f"{state['number']} is even"}

def odd(state):
    return {"result": f"{state['number']} is odd"}

def even_or_odd(state) -> str:     # 👈 router: returns a NODE NAME
    return "even" if state["number"] % 2 == 0 else "odd"

g = StateGraph(NumberState)
g.add_node("check", check)
g.add_node("even", even)
g.add_node("odd", odd)
g.add_edge(START, "check")
g.add_conditional_edges("check", even_or_odd, ["even", "odd"])   # list = possible destinations
g.add_edge("even", END)
g.add_edge("odd", END)
num_app = g.compile()
show_graph(num_app)

In [ ]:
for n in (4, 7):
    print(num_app.invoke({"number": n})["result"])

assert num_app.invoke({"number": 10})["result"] == "10 is even"
print("✅ Branching works!")

⚠️ **Router vs node:** a **node** returns a *state update* (`dict`). A **router** returns a *node name* (`str`). Routers must not change the state.

## Lesson 2 — Clean routers with `Literal`

Instead of passing the list of destinations, put them in the return type with `Literal`. The diagram then knows every branch. You can also branch **straight from `START`**.

In [ ]:
class Weather(TypedDict):
    temp: int
    advice: str

def hot(state):  return {"advice": "🥤 Drink water and stay inside"}
def mild(state): return {"advice": "🚶 Great day for a walk"}
def cold(state): return {"advice": "🧥 Wear a jacket"}

def by_temperature(state) -> Literal["hot", "mild", "cold"]:
    if state["temp"] >= 35:
        return "hot"
    if state["temp"] >= 18:
        return "mild"
    return "cold"

g = StateGraph(Weather)
for fn in (hot, mild, cold):
    g.add_node(fn.__name__, fn)
    g.add_edge(fn.__name__, END)
g.add_conditional_edges(START, by_temperature)    # branch right at the start
weather_app = g.compile()
show_graph(weather_app)

for t in (40, 25, 10):
    print(t, "°C →", weather_app.invoke({"temp": t})["advice"])

## Lesson 3 — Routers that return labels (path map)

Sometimes your router returns a *label* (`"yes"`, `"no"`) instead of a node name. Give a **path map** dict to translate labels → nodes:

```python
g.add_conditional_edges("check_age", is_adult, {"yes": "allow", "no": "deny"})
```

In [ ]:
class Person(TypedDict):
    age: int
    status: str

def check_age(state): return {}
def allow(state):     return {"status": "✅ Welcome!"}
def deny(state):      return {"status": "⛔ Sorry, 18+ only"}

def is_adult(state) -> str:
    return "yes" if state["age"] >= 18 else "no"

g = StateGraph(Person)
g.add_node("check_age", check_age)
g.add_node("allow", allow)
g.add_node("deny", deny)
g.add_edge(START, "check_age")
g.add_conditional_edges("check_age", is_adult, {"yes": "allow", "no": "deny"})
g.add_edge("allow", END)
g.add_edge("deny", END)
age_app = g.compile()
print(age_app.invoke({"age": 21})["status"])
print(age_app.invoke({"age": 15})["status"])

## Lesson 4 — Loops: go back to an earlier node

A conditional edge can point **backwards** — that's a loop. Agents work exactly like this: *think → act → check → think again …* until done.

```
START ──► roll_dice ──► got a six? ──no──┐
              ▲                          │
              └──────────────────────────┘
                         │ yes
                         ▼
                        END
```

We use a fixed list of dice rolls so the result is the same every time.

In [ ]:
class DiceState(TypedDict):
    rolls_to_use: list[int]              # pretend dice
    history: Annotated[list[int], operator.add]

def roll_dice(state):
    roll = state["rolls_to_use"][len(state["history"])]
    print("🎲 rolled", roll)
    return {"history": [roll]}

def got_six(state) -> Literal["roll_dice", "__end__"]:
    return END if state["history"][-1] == 6 else "roll_dice"   # END is the string "__end__"

g = StateGraph(DiceState)
g.add_node("roll_dice", roll_dice)
g.add_edge(START, "roll_dice")
g.add_conditional_edges("roll_dice", got_six)
dice_app = g.compile()
show_graph(dice_app)

res = dice_app.invoke({"rolls_to_use": [2, 5, 1, 6, 3], "history": []})
print("history:", res["history"])
assert res["history"] == [2, 5, 1, 6]
print("✅ Loop stopped at the first six!")

## Lesson 5 — Safety net: recursion limit

What if a loop never ends? LangGraph stops after **25 steps** by default and raises `GraphRecursionError`. You can change the limit with `config={"recursion_limit": N}`.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    # no six in the list for a long time → too many steps
    dice_app.invoke({"rolls_to_use": [1] * 50 + [6], "history": []},
                    config={"recursion_limit": 5})
except GraphRecursionError:
    print("🛑 Stopped by the recursion limit — your loop needs a better exit condition!")

💡 In real agents, also add your **own** counter (e.g. `attempts`) and stop after N tries. Don't rely only on the recursion limit.

## Lesson 6 — `Command`: update the state AND choose the next node

So far, a node updates the state and a separate router chooses the next node. With `Command`, **one node does both**:

```python
return Command(update={...}, goto="next_node")
```

Add a `Literal[...]` return type so LangGraph knows the possible destinations (for the diagram). You don't need `add_conditional_edges`.

In [ ]:
from langgraph.types import Command

class LoginState(TypedDict):
    password: str
    message: str

def check_password(state) -> Command[Literal["welcome", "reject"]]:
    if state["password"] == "langgraph123":
        return Command(update={"message": "password ok"}, goto="welcome")
    return Command(update={"message": "wrong password"}, goto="reject")

def welcome(state): return {"message": state["message"] + " → 🎉 logged in"}
def reject(state):  return {"message": state["message"] + " → 🔒 try again"}

g = StateGraph(LoginState)
g.add_node("check_password", check_password)
g.add_node("welcome", welcome)
g.add_node("reject", reject)
g.add_edge(START, "check_password")       # no conditional edges needed!
g.add_edge("welcome", END)
g.add_edge("reject", END)
login_app = g.compile()
show_graph(login_app)

print(login_app.invoke({"password": "langgraph123"})["message"])
print(login_app.invoke({"password": "1234"})["message"])

| Use… | When |
|---|---|
| `add_conditional_edges` | routing logic is separate from the work (easy to read & test) |
| `Command(goto=...)` | the node that does the work also knows where to go next (very common in multi-agent systems, Day 12) |

## Lesson 7 — Mini project: support-ticket triage

Combine everything: classify → set priority → route to a team → finalize.

```
START ► classify ► set_priority ─┬► billing ─┐
                                 ├► tech ────┼► finalize ► END
                                 └► general ─┘
```

In [ ]:
class Ticket(TypedDict):
    message: str
    category: str
    priority: str
    reply: str
    log: Annotated[list[str], operator.add]

def classify(state):
    text = state["message"].lower()
    if any(w in text for w in ("refund", "charge", "invoice")):
        cat = "billing"
    elif any(w in text for w in ("error", "crash", "bug")):
        cat = "tech"
    else:
        cat = "general"
    return {"category": cat, "log": [f"category={cat}"]}

def set_priority(state):
    urgent = any(w in state["message"].lower() for w in ("urgent", "asap"))
    p = "high" if urgent else "normal"
    return {"priority": p, "log": [f"priority={p}"]}

def route(state) -> Literal["billing", "tech", "general"]:
    return state["category"]

def billing(state): return {"reply": "Billing team is checking your payment.", "log": ["billing"]}
def tech(state):    return {"reply": "Tech team is looking at the bug.",        "log": ["tech"]}
def general(state): return {"reply": "Support will reply soon.",                "log": ["general"]}

def finalize(state):
    prefix = "🚨 HIGH PRIORITY: " if state["priority"] == "high" else ""
    return {"reply": prefix + state["reply"], "log": ["finalized"]}

g = StateGraph(Ticket)
for fn in (classify, set_priority, billing, tech, general, finalize):
    g.add_node(fn.__name__, fn)
g.add_edge(START, "classify")
g.add_edge("classify", "set_priority")
g.add_conditional_edges("set_priority", route)
for team in ("billing", "tech", "general"):
    g.add_edge(team, "finalize")
g.add_edge("finalize", END)
triage = g.compile()
show_graph(triage)

In [ ]:
r = triage.invoke({"message": "URGENT: app crashes with an error!", "log": []})
for k, v in r.items():
    print(f"{k:9}: {v}")

assert r["category"] == "tech" and r["priority"] == "high"
assert r["log"] == ["category=tech", "priority=high", "tech", "finalized"]
print("✅ Triage works!")

## Lesson 8 — Practice

**Exercise 1.** Grade a student: marks ≥ 75 → `distinction`, ≥ 40 → `pass`, else `fail`. Use a `Literal` router from `START`.

**Exercise 2 (loop).** Start with `n` and keep halving it (integer division) until it is ≤ 1. Store each value in `steps`. For `n=40` → `steps = [20, 10, 5, 2, 1]`.

In [ ]:
# ✅ Solution 1
class Marks(TypedDict):
    marks: int
    grade: str

def distinction(s): return {"grade": "distinction"}
def passed(s):      return {"grade": "pass"}
def fail(s):        return {"grade": "fail"}

def grade_router(s) -> Literal["distinction", "passed", "fail"]:
    return "distinction" if s["marks"] >= 75 else "passed" if s["marks"] >= 40 else "fail"

g = StateGraph(Marks)
for fn in (distinction, passed, fail):
    g.add_node(fn.__name__, fn)
    g.add_edge(fn.__name__, END)
g.add_conditional_edges(START, grade_router)
grade_app = g.compile()
assert [grade_app.invoke({"marks": m})["grade"] for m in (90, 50, 20)] == ["distinction", "pass", "fail"]

In [ ]:
# ✅ Solution 2
class Halve(TypedDict):
    n: int
    steps: Annotated[list[int], operator.add]

def halve(s):
    new = s["n"] // 2
    return {"n": new, "steps": [new]}

g = StateGraph(Halve)
g.add_node("halve", halve)
g.add_edge(START, "halve")
g.add_conditional_edges("halve", lambda s: END if s["n"] <= 1 else "halve", ["halve", END])
res = g.compile().invoke({"n": 40, "steps": []})
print(res)
assert res["steps"] == [20, 10, 5, 2, 1]
print("✅ All exercises solved!")

## 🎯 Day 3 Recap

* `add_conditional_edges(node, router)` — the router returns the **next node name**.
* Use `Literal[...]` or a path map so all branches are known.
* A route pointing backwards = **loop**. Always have an exit + `recursion_limit` as a safety net.
* `Command(update=..., goto=...)` updates state **and** routes in one node.

🎉 **You now know the whole core of LangGraph.** From tomorrow we put a real **LLM (OpenAI)** inside the nodes.

**Tomorrow (Day 4):** connect OpenAI using a `.env` file, and learn chat messages.